In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent   # /mnt/primary/Main
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.new_config import *

from uncertainty_prediction.baselines.deterministic.standard.base_regressor import RegressorConfig
from uncertainty_prediction.baselines.deterministic.standard import (
    RandomForestRuntimeRegressor,
    XGBoostRuntimeRegressor,
    LightGBMRuntimeRegressor,
    MLPRuntimeRegressor,
    GPRBFRuntimeRegressor,
)

set_seed(42)

In [2]:

"""
Load and extract query plans and features
"""
# any set of plans is fine here (... assuming that plans are identical across runs)
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260222-191819Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

In [3]:

"""
Split into training and testing sets
"""

train_qids, test_qids = split_query_ids(
    common,
    seed=SEED,
    test_frac=TEST_FRAC,
)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

runs_train, runs_test = make_train_test_run_split(
    runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
)


n_train: 351 n_test: 87


In [5]:

"""
Generate graph embeddings
"""

Z_train, Z_test, enc = make_plan_embeddings_by_run(
    encoder_cls=TrinoGraphWLPlanEncoder,
    plans_by_query=plans_by_query,
    runs_train=runs_train,
    runs_test=runs_test,
    train_qids=train_qids,
    test_qids=test_qids,
    encoder_kwargs={
        "emb_dim": 256,
        "n_iter": 4,
        "include_edge_types": True,
        "direction": "both",
        "standardise": True,
        "use_estimates": True,
        "use_cost_estimates": False,
    },
)


print("z dim:", next(iter(Z_train.values())).shape, "expected:", enc.dim())
print("n train runs:", len(Z_train), "n test runs:", len(Z_test))

z dim: (256,) expected: 256
n train runs: 1048 n test runs: 258


In [8]:
df_mean_baseline, mean_baseline_metrics, mean_baseline_info = evaluate_global_mean_runtime_baseline(
    runs_train,
    runs_test,
    xcol=XCOL,
)

keys = [
    "mae",
    "rmse",
    "spearman",
    "mean_q_error",
    "median_q_error",
    "p90_q_error",
    "p95_q_error",
]

print("\t".join(keys))
print("\t".join(f"{mean_baseline_metrics[k]:.4f}" for k in keys))

mae	rmse	spearman	mean_q_error	median_q_error	p90_q_error	p95_q_error
72.2048	230.4762	0.0000	111.4619	12.4905	410.8161	479.1371


/mnt/primary/Main/uncertainty_prediction/src/evaluate.py:643: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman = spearmanr(y_true, y_pred).correlation


In [23]:
config = RegressorConfig(
    log_target=True,
    log_base=2.0,
    random_state=SEED,
)

In [24]:

# model = XGBoostRuntimeRegressor(config=config)
# 
# model.tune(
#     X_train,
#     y_train,
#     search_type="random",
#     n_iter=12,
#     cv=3,
#     scoring="neg_mean_absolute_error",
# )
# 
# print(model.best_params_)
# print(model.evaluate(X_test, y_test))

best_params = {'subsample': 0.8, 'n_estimators': 256, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

model = XGBoostRuntimeRegressor(config=config, **best_params)
model.fit(X_train, y_train)

print_metrics_for_excel(model.evaluate(X_test, y_test))

53.4216	227.6522	0.7025	4.9930	2.2601	10.7685	24.3692


In [25]:

best_params = {'n_estimators': 500, 'min_samples_leaf': 2, 'max_depth': None}

model = RandomForestRuntimeRegressor(config=config, **best_params)
model.fit(X_train, y_train)

print_metrics_for_excel(model.evaluate(X_test, y_test))

53.2774	231.9781	0.7151	5.5099	2.1991	11.1040	20.9526


In [26]:

best_params = {'max_iter': 500, 'learning_rate_init': 0.01, 'hidden_layer_sizes': (256, 128, 64), 'alpha': 0.001}

model = MLPRuntimeRegressor(config=config, **best_params)
model.fit(X_train, y_train)

print_metrics_for_excel(model.evaluate(X_test, y_test))

56.3783	228.5144	0.6245	7.4706	2.2699	17.6278	38.7694


In [27]:

best_params = {'num_leaves': 15, 'n_estimators': 256, 'min_child_samples': 20, 'max_depth': 4, 'learning_rate': 0.05}

model = LightGBMRuntimeRegressor(config=config, **best_params)
model.fit(X_train, y_train)

print_metrics_for_excel(model.evaluate(X_test, y_test))

53.0753	224.8550	0.6815	5.4205	2.1237	10.5099	25.6029


/opt/miniconda3/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [28]:
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

best_params = {
    "gp__alpha": 1e-06,
    "gp__kernel": 1**2 * RBF(length_scale=10) + WhiteKernel(noise_level=0.01),
    "gp__n_restarts_optimizer": 0,
    "gp__normalize_y": True,
}

model = GPRBFRuntimeRegressor(config=config, use_scaler=True)

model.model = model._build_model()
model.model.set_params(**best_params)
model.model.fit(model._to_numpy(X_train), model._transform_target(y_train))

model.best_params_ = dict(best_params)
model.best_model_ = model.model
model.is_fitted = True

print_metrics_for_excel(model.evaluate(X_test, y_test))

53.7201	231.3679	0.6643	6.1316	2.4240	12.3559	20.3432
